In [1]:
# YouTube Comments Preprocessor
# Filters English comments, cleans text, prepares for sentiment labelling
# K-pop Sentiment Analysis Project

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Download required NLTK data
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

# --- Load raw comments ---
df = pd.read_csv("../01_raw_data/youtube/youtube_comments_raw.csv")
print(f"Raw comments loaded: {len(df)}")
print(df["group_comeback"].value_counts())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...


Raw comments loaded: 3000
group_comeback
aespa_Whiplash            500
IVE_RebelHeart            500
TWICE_Strategy            500
NCTDREAM_WhenImWithYou    500
SEVENTEEN_Thunder         500
StrayKids_ChkChkBoom      500
Name: count, dtype: int64


[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [2]:
from langdetect import detect, LangDetectException

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

# Filter English comments only
print("Filtering English comments...")
df["is_english"] = df["comment"].apply(is_english)
df_english = df[df["is_english"] == True].copy()
print(f"English comments: {len(df_english)} out of {len(df)}")
print(df_english["group_comeback"].value_counts())

Filtering English comments...
English comments: 1036 out of 3000
group_comeback
IVE_RebelHeart            196
aespa_Whiplash            176
SEVENTEEN_Thunder         174
TWICE_Strategy            173
NCTDREAM_WhenImWithYou    162
StrayKids_ChkChkBoom      155
Name: count, dtype: int64


In [3]:
# --- Text cleaning ---
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    # Lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)
    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned)
    # Remove stopwords and stem
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

print("Cleaning and preprocessing text...")
df_english["cleaned_comment"] = df_english["comment"].apply(clean_text)
df_english["processed_comment"] = df_english["comment"].apply(preprocess)

# Remove empty rows after cleaning
df_english = df_english[df_english["cleaned_comment"].str.len() > 5]

print(f"Comments after cleaning: {len(df_english)}")
print("\nSample cleaned comments:")
print(df_english[["comment", "cleaned_comment", "processed_comment"]].head(3).to_string())

Cleaning and preprocessing text...
Comments after cleaning: 1026

Sample cleaned comments:
                                                                                                                   comment                                                                                                   cleaned_comment                                 processed_comment
0                                                                                                                   Splash                                                                                                            splash                                            splash
5                                                                                                    I'm ready for it (cb)                                                                                                im ready for it cb                                             readi
6  the way I can immediately sense if the song i

In [4]:
# --- Save preprocessed data ---
df_english.to_csv("../02_processed_data/youtube_comments_processed.csv", index=False)
print(f"Saved {len(df_english)} preprocessed comments to 02_processed_data/youtube_comments_processed.csv")

# Also save a labelling template - just the columns needed for manual labelling
labelling_df = df_english[["group_comeback", "comment", "cleaned_comment"]].copy()
labelling_df["sentiment"] = ""  # Empty column for manual labels
labelling_df.to_csv("../03_labeled_data/youtube_comments_for_labelling.csv", index=False)
print(f"Saved labelling template to 03_labeled_data/youtube_comments_for_labelling.csv")

Saved 1026 preprocessed comments to 02_processed_data/youtube_comments_processed.csv
Saved labelling template to 03_labeled_data/youtube_comments_for_labelling.csv
